# 🎯 Entrenar el LoRA de defectos (SDXL / kohya) — Google Colab T4

**Turnkey:** descarga solo un dataset público de defectos (NEU-DET) y entrena el LoRA sin que subas nada. También acepta tus propias fotos.

Config **T4-safe** (768px, dim 16, AdamW8bit): un LoRA sólido en **~15–20 min** sin agotar la memoria de la T4 gratis.

**Uso:** `Entorno de ejecución → Cambiar tipo → T4 GPU`, luego *Ejecutar todo*.

## 1 · Verificar GPU (debe decir Tesla T4)

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Activá la GPU: Entorno de ejecución → Cambiar tipo → T4'
print('GPU:', torch.cuda.get_device_name(0))

## 2 · (Opcional) Montar Drive para guardar el LoRA

In [ ]:
USAR_DRIVE = True
DRIVE_BASE = '/content/drive/MyDrive/rollos_dataset'
import os
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print('Drive OK:', DRIVE_BASE)

## 3 · Instalar kohya sd-scripts

In [ ]:
%cd /content
import os
if not os.path.exists('/content/sd-scripts'):
    !git clone https://github.com/kohya-ss/sd-scripts
%cd /content/sd-scripts
!pip install -q -r requirements.txt
!pip install -q xformers bitsandbytes
print('sd-scripts listo.')

## 4 · Descargar el modelo base SDXL (RealVisXL V5.0)

In [ ]:
!apt-get -qq install -y aria2 >/dev/null
import os
BASE_CKPT = '/content/RealVisXL_V5.0_fp16.safetensors'
if not os.path.exists(BASE_CKPT):
    !aria2c -x16 -s16 -k1M --console-log-level=warn -d /content -o RealVisXL_V5.0_fp16.safetensors \
      https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors
print('Checkpoint base listo.')

## 5 · Conseguir los datos de entrenamiento
**Opción A (por defecto):** descarga el dataset público **NEU-DET** y lo recorta.
**Opción B:** si tenés tus fotos en Drive (`rollos_dataset/defectos_reales/<clase>/`), poné `USAR_NEU = False`.
`LIMIT` acota imágenes por clase (recomendado 80 en T4 gratis para que el cacheo no sature la RAM).

In [ ]:
USAR_NEU = True   # False = usar tus fotos desde Drive
TOKEN    = 'almdefect'
REPEATS  = 1
LIMIT    = 80     # imágenes por clase (80 = ligero y suficiente; 0 = todas)

import os, glob, subprocess
from PIL import Image

SRC = '/content/defectos_reales'
if USAR_NEU:
    cache = '/content/.neu'
    if not os.path.isdir(cache + '/IMAGES'):
        os.makedirs(cache, exist_ok=True)
        repo = 'https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection.git'
        run = lambda *a: subprocess.run(a, cwd=cache, check=True)
        run('git','init','-q'); run('git','remote','add','origin',repo)
        run('git','config','core.sparseCheckout','true')
        run('git','sparse-checkout','init','--no-cone')
        open(cache+'/.git/info/sparse-checkout','w').write('IMAGES/\n')
        run('git','fetch','--depth','1','--filter=blob:none','origin','master')
        run('git','checkout','-q','master')
    MAP = {'scratches':'scratch','pitted_surface':'corrosion','rolled-in_scale':'corrosion',
           'inclusion':'stain','patches':'stain','crazing':'crease'}
    n=0
    for p in sorted(glob.glob(cache+'/IMAGES/*.jpg')):
        base = os.path.basename(p).rsplit('.',1)[0]
        neu = next((c for c in MAP if base.startswith(c)), None)
        if not neu: continue
        d = os.path.join(SRC, MAP[neu]); os.makedirs(d, exist_ok=True)
        Image.open(p).convert('RGB').resize((512,512)).save(os.path.join(d, base+'.png')); n+=1
    print(f'{n} recortes NEU-DET en {SRC}')
else:
    SRC = os.path.join(DRIVE_BASE, 'defectos_reales')
    print('Usando tus fotos de', SRC)

## 6 · Preparar el dataset kohya (crops + captions)

In [ ]:
DST = '/content/lora_train'
folder = os.path.join(DST, f'{REPEATS}_{TOKEN}'); os.makedirs(folder, exist_ok=True)
PROMPT = {'scratch':'deep scratch scratches on brushed aluminum, metal scuff',
          'dent':'dent dented deformation on aluminum sheet, impact mark',
          'stain':'dark oil stain discoloration on aluminum',
          'crease':'crease wrinkle fold on aluminum foil',
          'corrosion':'corrosion oxidation pitting on aluminum',
          'tear':'torn ripped edge on aluminum foil'}
n=0
for clase, prompt in PROMPT.items():
    sub = os.path.join(SRC, clase)
    if not os.path.isdir(sub): continue
    vistos=0
    for p in sorted(glob.glob(sub+'/*')):
        if not p.lower().endswith(('.png','.jpg','.jpeg','.webp')): continue
        if LIMIT and vistos>=LIMIT: break
        vistos+=1
        try: im=Image.open(p).convert('RGB')
        except: continue
        im.thumbnail((1024,1024))
        b=f'{clase}_{n:04d}'
        im.save(os.path.join(folder,b+'.png'))
        open(os.path.join(folder,b+'.txt'),'w').write(f'{TOKEN}, {prompt}')
        n+=1
print(f'{n} imágenes listas en {folder}')
assert n>0, 'Sin imágenes: revisá la celda 5 (datos).'

## 7 · Entrenar el LoRA (config T4-safe)
768px · dim 16 · AdamW8bit · 600 pasos. Verás `caching latents` y luego los `steps` avanzando (~15–20 min).
Si viste que se cortaba tras *"Using DreamBooth method"*, era por config demasiado pesada; esta es ligera.

In [ ]:
# verificación rápida del dataset antes de entrenar
!echo "imágenes:" && ls /content/lora_train/1_almdefect/*.png 2>/dev/null | wc -l
!echo "captions:" && ls /content/lora_train/1_almdefect/*.txt 2>/dev/null | wc -l

!accelerate launch --num_cpu_threads_per_process 2 sdxl_train_network.py \
  --pretrained_model_name_or_path="/content/RealVisXL_V5.0_fp16.safetensors" \
  --train_data_dir="/content/lora_train" \
  --output_dir="/content/output" \
  --output_name="rollo_defectos_lora" \
  --resolution="768,768" \
  --network_module=networks.lora \
  --network_dim=16 --network_alpha=8 \
  --train_batch_size=1 \
  --max_train_steps=600 \
  --learning_rate=1e-4 \
  --optimizer_type=AdamW8bit \
  --lr_scheduler=cosine --lr_warmup_steps=30 \
  --gradient_checkpointing \
  --lowram \
  --network_train_unet_only \
  --cache_latents \
  --enable_bucket --min_bucket_reso=256 --max_bucket_reso=1024 \
  --mixed_precision=fp16 --save_precision=fp16 \
  --xformers \
  --caption_extension=.txt \
  --save_model_as=safetensors \
  --seed=42

## 8 · Guardar / descargar el LoRA

In [ ]:
import shutil, glob, os
finales = glob.glob('/content/output/*.safetensors')
print('LoRAs generados:', finales)
assert finales, 'VACÍO: el entrenamiento no guardó. Revisá el final de la celda 7.'
if USAR_DRIVE:
    dst = os.path.join(DRIVE_BASE, 'loras'); os.makedirs(dst, exist_ok=True)
    for f in finales: shutil.copy(f, dst)
    print('Copiado a Drive:', dst)
else:
    from google.colab import files
    files.download(sorted(finales)[-1])
print('Poné rollo_defectos_lora.safetensors en runtime/ComfyUI/models/loras/ (local).')